### Reranking Hybrid Search Statergies

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.

👉 It ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [70]:
# Correct way to import inside your Python scripts
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser


In [71]:
## load text file
loader=TextLoader("langchain_sample.txt")
raw_docs=loader.load()

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
docs


[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.\nBM25 and vector-based retrieval can be combined in LangChain to support hybrid retrieval strategies.'),
 Docu

In [72]:
## user query
query="How can i use langchain to build an application with memory and tools?"

In [73]:
### FAISS and Huggingface model Embeddings

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embedding_model)
retriever=vectorstore.as_retriever(search_kwargs={"k":8})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4656.67it/s]


In [74]:
## Hugging Face Embedding
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore_openai=FAISS.from_documents(docs,embeddings)
retriever_openai=vectorstore_openai.as_retriever(search_kwargs={"k":8})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3765.61it/s]


In [75]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017FF2546750>, search_kwargs={'k': 8})

In [76]:
retriever_openai

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017FEE5EB5F0>, search_kwargs={'k': 8})

In [77]:
## prompt and use the llm
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0)
llm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017FDEE2B1A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017FEE574A70>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [78]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [79]:
retrieved_docs=retriever.invoke(query)
retrieved_docs

[Document(id='d0c542da-462a-4fd0-9c77-a3f6c480f00c', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.'),
 Document(id='bc41e30e-cff7-40f7-b7c1-b1f3616173b9', metadata={'source': 'langchain_sample.txt'}, page_content="The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.\nThe 'refine' chain iteratively updates an answer by incorporating each new chunk of information.\nLangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summa

In [80]:
chain=prompt| llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user\'s question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput format: comma-separated document indices (e.g., 2,1,3,0,...)\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video

In [81]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [82]:
doc_lines

['1. LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.',
 "2. The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.\nThe 'refine' chain iteratively updates an answer by incorporating each new chunk of information.\nLangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.",
 '3. LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps 

In [83]:
formatted_docs

"1. LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\n2. The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.\nThe 'refine' chain iteratively updates an answer by incorporating each new chunk of information.\nLangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.\n3. LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.\nMemory in LangChain helps models 

In [84]:
response=chain.invoke({"question":query,"documents":formatted_docs})
response

'To determine the relevance of each document to the user\'s question, "How can I use LangChain to build an application with memory and tools?", let\'s analyze the key points from each document:\n\n1. **Introduction to LangChain**: This document provides a general overview of LangChain, mentioning its support for memory and agents but lacks specific details on how to build an application with these features.\n2. **Chains and Memory**: This document discusses specific types of chains and how LangChain supports conversational memory and summarization memory, directly addressing the "memory" aspect of the question. It also touches upon agents deciding which tools to use.\n3. **Retrieval-Augmented Generation (RAG) and Memory**: This document explains how LangChain enables RAG, the use of memory for multi-turn conversations, and how agents can use tools, directly addressing both "memory" and "tools" in the context of application development.\n4. **Agents, APIs, and Tool Usage**: While this d

In [85]:
# Step 5: Parse and rerank
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
indices

[1, 3, 0, 5, 5]

In [86]:
retrieved_docs

[Document(id='d0c542da-462a-4fd0-9c77-a3f6c480f00c', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.\nThe framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.'),
 Document(id='bc41e30e-cff7-40f7-b7c1-b1f3616173b9', metadata={'source': 'langchain_sample.txt'}, page_content="The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.\nThe 'refine' chain iteratively updates an answer by incorporating each new chunk of information.\nLangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summa

In [87]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='bc41e30e-cff7-40f7-b7c1-b1f3616173b9', metadata={'source': 'langchain_sample.txt'}, page_content="The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.\nThe 'refine' chain iteratively updates an answer by incorporating each new chunk of information.\nLangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory."),
 Document(id='7c43b8b9-c391-42f0-9bd7-0e55bebe03fc', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.\nMMR (Maximal Marginal Relevance) retrieval in LangChain improves diversity by balancing 

In [88]:
# Step 6: Show results
print("\n📊 Final Reranked Results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


📊 Final Reranked Results:

Rank 1:
The 'map-reduce' chain breaks up large documents, processes them separately, and then aggregates the outputs.
The 'refine' chain iteratively updates an answer by incorporating each new chunk of information.
LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.
LangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.

Rank 2:
LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.
RAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.
MMR (Maximal Marginal Relevance) retrieval in LangChain improves diversity by balancing relevance and redundancy.
Tool usage in LangChain allows agents to execute predefined Python functions with contextual input from the user.

Rank 3:
LangChain is an open-source framework 